# 00. Подготовка данных

Скачивает и готовит все датасеты на Google Drive. Запускается **один раз**, дальше остальные ноутбуки берут данные с Drive.

| Датасет | Для чего | Что хранится на Drive |
| --- | --- | --- |
| Golos (test) | речь с расшифровками для модуля 1 | ~1 ГБ |
| MUSAN (noise, music) | фон для смесей | до 2 ГБ |
| Dusha (Hugging Face) | эмоции, модуль 2 | 2–4 ГБ |
| Speech Commands v0.02 | короткие слова, модуль 3 | ~0,3 ГБ (10 слов) |

Перед запуском: **Среда выполнения → Сменить среду → T4 GPU** (для подготовки данных GPU не нужен, но так не придётся перезапускаться).

In [ ]:
# Подключаем Google Drive, скачиваем код и ставим зависимости (~2 минуты)
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/tanyalebedeva/call-audio-analysis.git'
!test -d /content/call-audio-analysis || git clone -q $REPO /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
!mkdir -p $DATA $RUNS
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU не подключён: Среда выполнения → Сменить среду → T4 GPU'

## Проверка кода на синтетике
За ~1 минуту проверяет, что обучение, чекпойнты и метрики работают в этой среде.

In [ ]:
!python -m pytest -q

## 1. Golos — речь с расшифровками

In [ ]:
!python -m callaudio.data.prepare golos --out $DATA

## 2. MUSAN — шум и музыка
Архив ~11 ГБ скачивается во временную папку Colab, на Drive кладётся только нужное.

In [ ]:
!python -m callaudio.data.prepare musan --out $DATA --music-files 300

## 3. Dusha — эмоции
6000 записей на класс в train (если в классе меньше — берутся все) и 1000 в test.

In [ ]:
!python -m callaudio.data.prepare dusha --out $DATA --per-class 6000 --test-per-class 1000

## 4. Окна Dusha по 1 с — для VAE на русской речи (модуль 3 → признаки для модуля 2)

In [ ]:
!python -m callaudio.data.prepare dusha_windows --out $DATA

## 5. Speech Commands — короткие слова для VAE

In [ ]:
!python -m callaudio.data.prepare commands --out $DATA --words core

## 6. Анализ данных — графики для главы 2 записки

In [ ]:
import json, collections, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
M = Path(DATA) / 'manifests'
def load(name): return [json.loads(l) for l in open(M / name, encoding='utf-8')]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
emo = load('emotion_train.jsonl')
cnt = collections.Counter(r['label'] for r in emo)
axes[0].bar(cnt.keys(), cnt.values()); axes[0].set_title('Dusha (train): записей по эмоциям')
axes[1].hist([r['duration'] for r in emo], bins=40); axes[1].set_title('Dusha: длительность, с')
speech = load('speech_train.jsonl')
axes[2].hist([r['duration'] for r in speech], bins=40); axes[2].set_title('Golos: длительность, с')
plt.tight_layout(); plt.savefig(f'{DATA}/data_overview.png', dpi=150); plt.show()
for name in sorted(p.name for p in M.glob('*.jsonl')):
    rows = load(name)
    print(f"{name:32s} {len(rows):7d} записей  {sum(r.get('duration', 0) for r in rows) / 3600:6.2f} ч")

In [ ]:
# Примеры спектрограмм: представления аудио для главы 1
import torch
from callaudio.audio import load_audio, LogMel, EMOTION_MEL, vocoder_mel
row = next(r for r in emo if r['label'] == 'angry')
wav = load_audio(row['path'])[:16000 * 3]
fig, axes = plt.subplots(3, 1, figsize=(10, 7))
axes[0].plot(np.arange(len(wav)) / 16000, wav, lw=0.5); axes[0].set_title('Waveform')
spec = torch.stft(torch.from_numpy(wav), 512, 128, window=torch.hann_window(512), return_complex=True).abs()
axes[1].imshow(torch.log(spec + 1e-5), origin='lower', aspect='auto', cmap='magma'); axes[1].set_title('Спектрограмма STFT (лог-амплитуда)')
axes[2].imshow(LogMel(EMOTION_MEL)(torch.from_numpy(wav)[None])[0], origin='lower', aspect='auto', cmap='magma'); axes[2].set_title('Лог-мел-спектрограмма, 64 полосы')
plt.tight_layout(); plt.savefig(f'{DATA}/representations.png', dpi=150); plt.show()